# Task 1 — Rating prediction (latent factor model)

Explore and pick hyperparameters here; copy the winner into `src/assignment_1/tasks/task1.py`.

HW3 target: public MSE ≤ 0.7280 (baseline 0.7429).

In [10]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import numpy as np
import pandas as pd
from surprise import BaselineOnly
from tqdm.auto import tqdm

from assignment_1.modules.models import (
    fit_bias,
    fit_svd,
    predict_bias,
    predict_svd,
    to_surprise_trainset,
)
from assignment_1.modules.util import (
    calc_mse,
    drop_zero_ratings,
    load_csv,
    random_split,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
TASK_TRAIN_FILE = "tasks_1_2_train.csv"
TASK_TEST_FILE = "task_1_test.csv"
train = load_csv(TASK_TRAIN_FILE, DATA_DIR)
test = load_csv(TASK_TEST_FILE, DATA_DIR)

## Look at the data
Check the rating range: Surprise clips predictions to it, and the baseline assumed 1–5.

In [3]:
print(
    f"{len(train):,} ratings | {train.user.nunique():,} users | {train.item.nunique():,} items"
)
print(
    f"test users unseen in train: {(~test.user.isin(train.user)).mean():.1%}, items: {(~test.item.isin(train.item)).mean():.1%}"
)

train.rating.value_counts().sort_index()

1,861,643 ratings | 30,909 users | 51,007 items
test users unseen in train: 0.0%, items: 0.0%


rating
0     27626
1     36443
2     97748
3    314624
4    641182
5    744020
Name: count, dtype: int64

## Validation split + baseline yardstick
Same split for every model, so differences come from the model, not the split. `rating_scale` comes from the *full* training data so validation-split models clip the same way the final model does.

In [4]:
tr, val = random_split(train, val_frac=0.25, seed=0)
tr = drop_zero_ratings(tr)
val = drop_zero_ratings(val)
 
SCALE = (train.rating.min(), train.rating.max())


def val_mse(**params):
    model = fit_svd(tr, rating_scale=SCALE, **params)
    return calc_mse(val.rating, predict_svd(model, val.user, val.item))


base = val_mse(n_factors=1, n_epochs=50)
print(f"baseline val MSE = {base:.4f}  ->  2% target ≈ {base * 0.98:.4f}")

baseline val MSE = 0.6021  ->  2% target ≈ 0.5900


## Step 1: sweep regularization only (λ = `reg_all`)
If the best value is at the edge of the range, extend the range.

In [5]:
grid = {"n_factors": [1, 10, 50], "reg_all": [0.1, 0.3, 0.5], "n_epochs": [50]}

rows = []
for values in tqdm(
    itertools.product(*grid.values()), total=prod(map(len, grid.values()))
):
    params = dict(zip(grid, values))
    s = val_mse(**params)
    rows.append({**params, "val_mse": s, "vs_base_%": (1 - s / base) * 100})

pd.DataFrame(rows).sort_values("val_mse")

  0%|          | 0/9 [00:00<?, ?it/s]

,n_factors,reg_all,n_epochs,val_mse,vs_base_%
0,1,0.1,50,0.601135,0.153276
3,10,0.1,50,0.601460,0.099343
6,50,0.1,50,0.601822,0.039151
1,1,0.3,50,0.617347,-2.539444
4,10,0.3,50,0.617396,-2.547646
7,50,0.3,50,0.617550,-2.573190
2,1,0.5,50,0.636800,-5.770524
5,10,0.5,50,0.636823,-5.774468
8,50,0.5,50,0.636901,-5.787299


In [7]:
grid = {"n_factors": [1], "reg_all": [0.02, 0.05, 0.1], "n_epochs": [20, 50]}

rows = []
for values in tqdm(
    itertools.product(*grid.values()), total=prod(map(len, grid.values()))
):
    params = dict(zip(grid, values))
    s = val_mse(**params)
    rows.append({**params, "val_mse": s, "vs_base_%": (1 - s / base) * 100})

pd.DataFrame(rows).sort_values("val_mse")

  0%|          | 0/6 [00:00<?, ?it/s]

,n_factors,reg_all,n_epochs,val_mse,vs_base_%
3,1,0.05,50,0.598900,0.524515
5,1,0.10,50,0.601135,0.153276
1,1,0.02,50,0.602058,0.000000
0,1,0.02,20,0.604109,-0.340771
2,1,0.05,20,0.605419,-0.558221
4,1,0.10,20,0.608087,-1.001361


## Next
- Sweep `n_factors`, `lr_all`, `n_epochs` around the best λ.
- Copy the winner into `TASK1_SVD_PARAMS` in `tasks/task1.py`, then `make predict` (or `make submit`).

In [8]:
def val_mse_bias(reg_u, reg_i, n_epochs=10):
    algo = BaselineOnly(
        bsl_options={
            "method": "als",
            "reg_u": reg_u,
            "reg_i": reg_i,
            "n_epochs": n_epochs,
        },
        verbose=False,
    )
    algo.fit(to_surprise_trainset(tr, SCALE))
    preds = [algo.predict(u, i).est for u, i in zip(val.user, val.item)]

    return calc_mse(val.rating, preds)


grid = {"reg_u": [0.3, 1, 3], "reg_i": [2, 5, 10], "n_epochs": [25]}

rows = []
for ru, ri, ep in tqdm(
    itertools.product(*grid.values()), total=prod(map(len, grid.values()))
):
    s = val_mse_bias(ru, ri, n_epochs=ep)
    rows.append(
        {
            "reg_u": ru,
            "reg_i": ri,
            "n_epochs": ep,
            "val_mse": s,
            "vs_base_%": (1 - s / base) * 100,
        }
    )

results = pd.DataFrame(rows).sort_values("val_mse")
results.head(10)

  0%|          | 0/9 [00:00<?, ?it/s]

,reg_u,reg_i,n_epochs,val_mse,vs_base_%
4,1.0,5,25,0.594096,1.322368
1,0.3,5,25,0.594210,1.303461
3,1.0,2,25,0.594400,1.271865
0,0.3,2,25,0.594494,1.256280
7,3.0,5,25,0.596277,0.960213
6,3.0,2,25,0.596642,0.899606
5,1.0,10,25,0.596690,0.891597
2,0.3,10,25,0.596878,0.860287
8,3.0,10,25,0.598680,0.560980


In [11]:
bias_model = fit_bias(tr, reg_u=1.0, reg_i=5.0, n_epochs=25, rating_scale=SCALE)
svd_model = fit_svd(tr, rating_scale=SCALE, n_factors=1, reg_all=0.05, n_epochs=50)

b = predict_bias(bias_model, val.user, val.item)
s = predict_svd(svd_model, val.user, val.item)

print(f"bias alone: {calc_mse(val.rating, b):.4f}   (expect 0.5941)")
print(f"svd alone:  {calc_mse(val.rating, s):.4f}   (expect 0.5989)")

for w in np.linspace(0, 1, 11):
    print(f"w={w:.1f}  MSE={calc_mse(val.rating, w * b + (1 - w) * s):.4f}")

bias alone: 0.5941   (expect 0.5941)
svd alone:  0.5989   (expect 0.5989)
w=0.0  MSE=0.5989
w=0.1  MSE=0.5979
w=0.2  MSE=0.5970
w=0.3  MSE=0.5962
w=0.4  MSE=0.5955
w=0.5  MSE=0.5950
w=0.6  MSE=0.5946
w=0.7  MSE=0.5943
w=0.8  MSE=0.5941
w=0.9  MSE=0.5940
w=1.0  MSE=0.5941
